# 🎬 Cloud Video Generator (Wan 2.2 Optimized)
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### High-Performance T2V & I2V for T4 GPUs

This notebook implements the **Alibaba Wan-AI** video generation pipeline. 

**Features:**
- **Model:** Wan 2.1/2.2-T2V-1.3B (Optimized for 16GB VRAM).
- **Optimizations:** FP16, Xformers, Model CPU Offloading, and Attention Slicing.
- **UI:** Gradio-powered Web Interface for Text-to-Video and Image-to-Video.
- **Runtime:** Fully compatible with Colab (T4) and Kaggle (T4).

## 🛠️ Step 1: Environment Setup
Install all necessary dependencies for the Wan architecture and Gradio UI.

In [ ]:
# Update and install required libraries
!pip install -q -U diffusers transformers accelerate gradio xformers moviepy sentencepiece

## 🧠 Step 2: Model Configuration & Loading
We use `WanPipeline` with `enable_model_cpu_offload()` to ensure the massive T5 text encoder and the video diffusion model fit within the T4's 16GB VRAM limit.

In [ ]:
import torch
import gc
import os
import uuid
from diffusers import WanPipeline, WanImageToVideoPipeline
from diffusers.utils import export_to_video
import gradio as gr
from PIL import Image

# Global variables for pipelines
t2v_pipe = None
i2v_pipe = None

# Model IDs (Using 1.3B for T4 stability)
T2V_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
I2V_MODEL_ID = "Wan-AI/Wan2.1-I2V-1.3B-Diffusers"

def clear_memory():
    """Clears GPU cache and garbage collects to prevent OOM."""
    gc.collect()
    torch.cuda.empty_cache()

def load_pipeline(mode="t2v"):
    global t2v_pipe, i2v_pipe
    clear_memory()
    
    if mode == "t2v":
        if t2v_pipe is None:
            print(f"Loading T2V Model: {T2V_MODEL_ID}...")
            t2v_pipe = WanPipeline.from_pretrained(
                T2V_MODEL_ID, 
                torch_dtype=torch.float16, 
                use_safetensors=True
            )
            # Optimization suite for T4
            t2v_pipe.enable_model_cpu_offload()
            t2v_pipe.enable_attention_slicing()
            try:
                t2v_pipe.enable_xformers_memory_efficient_attention()
            except Exception as e:
                print("Xformers not available, using default attention.")
        return t2v_pipe
    else:
        if i2v_pipe is None:
            print(f"Loading I2V Model: {I2V_MODEL_ID}...")
            i2v_pipe = WanImageToVideoPipeline.from_pretrained(
                I2V_MODEL_ID, 
                torch_dtype=torch.float16,
                use_safetensors=True
            )
            i2v_pipe.enable_model_cpu_offload()
            i2v_pipe.enable_attention_slicing()
        return i2v_pipe

print("✅ Setup logic initialized. Run the next cell to launch UI.")

## 🎨 Step 3: Interactive Web UI
This cell launches a Gradio interface. You can generate videos from text or animate existing images.

In [ ]:
def generate_t2v(prompt, negative_prompt, steps, guidance, seed):
    try:
        pipe = load_pipeline("t2v")
        generator = torch.Generator("cuda").manual_seed(seed) if seed != -1 else None
        
        # Optimal resolution for 1.3B is 480x832
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_frames=81, 
            height=480,
            width=832,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator
        ).frames[0]
        
        output_path = f"t2v_{uuid.uuid4().hex}.mp4"
        export_to_video(output, output_path, fps=16)
        return output_path
    except Exception as e:
        return f"Error: {str(e)}"

def generate_i2v(image, prompt, steps, guidance, seed):
    try:
        if image is None: return "Error: Please upload an image."
        pipe = load_pipeline("i2v")
        generator = torch.Generator("cuda").manual_seed(seed) if seed != -1 else None
        
        output = pipe(
            image=image,
            prompt=prompt,
            num_frames=81,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator
        ).frames[0]
        
        output_path = f"i2v_{uuid.uuid4().hex}.mp4"
        export_to_video(output, output_path, fps=16)
        return output_path
    except Exception as e:
        return f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎥 Cloud Video Generator")
    
    with gr.Tab("Text to Video"):
        with gr.Row():
            with gr.Column():
                t2v_prompt = gr.Textbox(label="Prompt", placeholder="A futuristic cyberpunk city with neon lights...")
                t2v_neg = gr.Textbox(label="Negative Prompt", value="low quality, blurry, static")
                with gr.Row():
                    t2v_steps = gr.Slider(10, 50, value=30, step=1, label="Steps")
                    t2v_guidance = gr.Slider(1.0, 15.0, value=6.0, step=0.5, label="Guidance Scale")
                t2v_seed = gr.Number(label="Seed (-1 for random)", value=-1)
                t2v_btn = gr.Button("Generate Video", variant="primary")
            with gr.Column():
                t2v_output = gr.Video(label="Result")
                
    with gr.Tab("Image to Video"):
        with gr.Row():
            with gr.Column():
                i2v_input = gr.Image(type="pil", label="Input Image")
                i2v_prompt = gr.Textbox(label="Prompt (Optional)", placeholder="Make the character smile and wave...")
                with gr.Row():
                    i2v_steps = gr.Slider(10, 50, value=30, step=1, label="Steps")
                    i2v_guidance = gr.Slider(1.0, 15.0, value=6.0, step=0.5, label="Guidance Scale")
                i2v_seed = gr.Number(label="Seed (-1 for random)", value=-1)
                i2v_btn = gr.Button("Animate Image", variant="primary")
            with gr.Column():
                i2v_output = gr.Video(label="Result")
    
    t2v_btn.click(generate_t2v, inputs=[t2v_prompt, t2v_neg, t2v_steps, t2v_guidance, t2v_seed], outputs=t2v_output)
    i2v_btn.click(generate_i2v, inputs=[i2v_input, i2v_prompt, i2v_steps, i2v_guidance, i2v_seed], outputs=i2v_output)

demo.launch(share=True, debug=True)

## 📺 Step 4: Video Preview & Cleanup
Run this cell to view the last generated video directly in the notebook and manage local storage.

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob

# Find the latest generated mp4
list_of_files = glob.glob('*.mp4')
if list_of_files:
    latest_file = max(list_of_files, key=os.path.getctime)
    print(f"Displaying latest video: {latest_file}")
    
    mp4 = open(latest_file,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=600 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("No videos found. Generate one in the UI first!")

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*